## Setup and imports

In [ ]:
EXP_NAME = "SPS"

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
  from google.colab import userdata
  from google.colab import drive
  drive.mount('/content/drive')
  PROJECT_ROOT = userdata.get('PROJECT_ROOT')
else:
  PROJECT_ROOT = '../..'

In [ ]:
print(PROJECT_ROOT)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1.2)

sps_audit = pd.read_csv(f'{PROJECT_ROOT}/results/{EXP_NAME}/sps_audit.csv')
sbs_audit_baseline = pd.read_csv(f'{PROJECT_ROOT}/results/{EXP_NAME}/sps_audit_baseline.csv')
dataset = pd.read_csv(f'{PROJECT_ROOT}/data/heart_disease_cleaned.csv')

In [ ]:
# sps_audit.head()

# Analysis

### Basics

In [ ]:
scorings = sps_audit.groupby('feature')[['sensitivity_scoring', 'fidelity_scoring']].first()

sps_audit = sps_audit.drop(['sensitivity_scoring', 'fidelity_scoring'], axis=1)

In [ ]:
iteration_per_feature = sps_audit[sps_audit['bucket'] == 'x_desc'].groupby('feature')[['roc_auc']].count()
iteration_per_feature

### Counterfactual sensitivity

In [ ]:
sps_audit.groupby('feature')['cf_sensitivity'].agg(['mean','std','median']).sort_values(by='mean')

In [ ]:
# Counterfactual sensitivity
f, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(data=sps_audit, x="feature", y="cf_sensitivity", showmeans=True, 
            meanprops={"marker": "+",
                       "markeredgecolor": "black",
                       "markersize": "8"}, ax=ax)

plt.plot()


# Trade-off analyses

In [ ]:
group_ieco_mace_cols = sps_audit.columns.str.startswith('ieco_mace_')
group_ieco_mace_baseline_cols = sbs_audit_baseline.columns.str.startswith('ieco_mace_')
sps_audit['max_ieco_mace'] = sps_audit.loc[:, group_ieco_mace_cols].max(axis=1)
sbs_audit_baseline['max_ieco_mace'] = sbs_audit_baseline.loc[:, group_ieco_mace_baseline_cols].max(axis=1)

group_total_mace_cols = sps_audit.columns.str.startswith('total_mace_')
group_total_mace_baseline_cols = sbs_audit_baseline.columns.str.startswith('total_mace_')
sps_audit['max_total_mace'] = sps_audit.loc[:, group_total_mace_cols].max(axis=1)
sbs_audit_baseline['max_total_mace'] = sbs_audit_baseline.loc[:, group_total_mace_baseline_cols].max(axis=1)

In [ ]:

feature_desc_stats = sps_audit[sps_audit['bucket'] == 'x_desc'].groupby(['feature'])[['total_mace', 'max_total_mace', 'auprc', 'roc_auc', 'te_error']].median()
feature_corr_stats = sps_audit[sps_audit['bucket'] == 'x_corr'].groupby(['feature'])[['total_mace', 'max_total_mace','auprc', 'roc_auc', 'te_error']].median()

In [ ]:
baseline_te_error = sbs_audit_baseline['te_error'].values[0]
baseline_total_mace = sbs_audit_baseline['total_mace'].values[0]
baseline_max_total_mace = sbs_audit_baseline['max_total_mace'].values[0]
baseline_roc_auc = sbs_audit_baseline['roc_auc'].values[0]
baseline_auprc = sbs_audit_baseline['auprc'].values[0]

## Total Effect estimation vs Fairness trade-off

In [ ]:
print(f"Baseline TE error: {baseline_te_error}")
print(f"Baseline MACE: {baseline_total_mace}")
print(f"Baseline maximum group MACE: {baseline_max_total_mace}")

In [ ]:
te_error_vs_fairness = sps_audit.groupby('iteration')[['te_error', 'max_total_mace', 'auprc', 'ieco_mace']].first().sort_values('max_total_mace')
x_desc_configs = sps_audit[sps_audit['bucket'] == 'x_desc'].groupby('iteration')['feature'].apply(list).to_dict()

pareto_frontier = []
current_min_te_error = 1

print('--- Configurations on the TE error Pareto Frontier ---')
for idx, solution in te_error_vs_fairness.iterrows():
  if solution['te_error'] < current_min_te_error:
    pareto_frontier.append(solution)
    current_min_te_error = solution['te_error']
    print(f'Iteration {idx}, Xdesc: {x_desc_configs[idx]}')
pareto_frontier_df = pd.DataFrame(pareto_frontier)
print(pareto_frontier_df.to_markdown())


In [ ]:


fig, axes = plt.subplots(1, 2,  figsize=(20, 6))
axes[0].plot(baseline_te_error, baseline_total_mace, marker="D", color="#D92B89", linestyle="")
axes[1].plot(baseline_te_error, baseline_total_mace, marker="D", color="#D92B89", linestyle="")
sns.scatterplot(data=te_error_vs_fairness, x='te_error', y='max_total_mace', size="auprc", sizes=(20, 200) ,ax=axes[0])
axes[0].legend_.set_title('AUPRC')
sns.lineplot(data=pareto_frontier_df, x='te_error', y='max_total_mace', color="green", marker='', linestyle="--", errorbar=None, ax=axes[0])
sns.lineplot(data=pareto_frontier_df, x='te_error', y='max_total_mace', color="green", linestyle="--", errorbar=None, ax=axes[1])
sns.scatterplot(data=feature_desc_stats, x='te_error', y='max_total_mace', hue='feature', color='', ax=axes[1])
# plt.axvline(auprc_baseline)
plt.xlabel('TE Error')
plt.ylabel('Max group MACE')
plt.legend(labels=[ 
                   'Bias-unaware configuration',
                   'Pareto frontier', 
                   "Feature config results",
                   ]+feature_desc_stats.index.to_list(),
           loc='upper left', bbox_to_anchor=(1, 1), edgecolor="white")

plt.show()

# Utility vs Fairness trade-off

In [ ]:
print(f"Baseline Average Precision: {baseline_auprc}")
print(f"Baseline ROC AUC: {baseline_roc_auc}")
print(f"Baseline max MACE: {baseline_max_total_mace}")

## Pareto Frontier 


In [ ]:
utility_vs_fairness = sps_audit.groupby('iteration')[['roc_auc', 'auprc','total_mace', 'max_total_mace', 'te_error']].first().sort_values('max_total_mace')
x_desc_configs = sps_audit[sps_audit['bucket'] == 'x_desc'].groupby('iteration')['feature'].apply(list).to_dict()

positives = dataset[dataset['cvd'] == 1]
auprc_baseline = len(positives) / len(dataset)

pareto_frontier = []
current_max_utility = -1

print('--- Configurations on the AUPRC Pareto Frontier ---')
for idx, solution in utility_vs_fairness.iterrows():
  if solution['auprc'] > current_max_utility:
    pareto_frontier.append(solution)
    current_max_utility = solution['auprc']
    print(f'Iteration {idx}, Xdesc: {x_desc_configs[idx]}')
pareto_frontier_df = pd.DataFrame(pareto_frontier)
print(pareto_frontier_df.to_markdown())


fig, axes = plt.subplots(1, 2,  figsize=(20, 6))
axes[0].plot(baseline_roc_auc, baseline_total_mace, marker="D", color="#D92B89", linestyle="")
axes[1].plot(baseline_roc_auc, baseline_total_mace, marker="D", color="#D92B89", linestyle="")
sns.lineplot(data=pareto_frontier_df, x='auprc', y='max_total_mace', color="green", marker='', linestyle="--", errorbar=None, ax=axes[0])
sns.lineplot(data=pareto_frontier_df, x='auprc', y='max_total_mace', color="green", linestyle="--", errorbar=None, ax=axes[1])
sns.scatterplot(data=utility_vs_fairness, x='auprc', y='max_total_mace', ax=axes[0])
sns.scatterplot(data=feature_desc_stats, x='auprc', y='max_total_mace', hue='feature', color='', ax=axes[1])
# plt.axvline(auprc_baseline)
plt.xlabel('Average Precision (AUPRC)')
plt.ylabel('Max group MACE')
plt.legend(labels=[ 
                   'Bias-unaware configuration',
                   'Pareto frontier', 
                   'CEVAE-HE result for each pathway configuration in the audit'
                   ]+feature_desc_stats.index.to_list(),
           loc='upper left', bbox_to_anchor=(1, 1), edgecolor="white")

plt.show()